# Loading libraries


In [ ]:
import pandas as pd
import numpy as np
import sklearn

# Loading datasets

### Labeled ClinPGx

In [ ]:
labeled_clinpgx_dataset = pd.read_csv('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/labeled_clinpgx_dataset.csv',
                               sep="\t")

labeled_clinpgx_dataset = labeled_clinpgx_dataset.drop(columns='User_input_1')

labeled_clinpgx_dataset.drop_duplicates(inplace=True)


### Unlabeled Populational

In [ ]:
unlabeled_populational_dataset = pd.read_csv('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/unlabeled_populational_dataset.csv',
                               sep="\t")

unlabeled_populational_dataset = unlabeled_populational_dataset.drop(columns='User_input_1')

unlabeled_populational_dataset.drop_duplicates(inplace=True)


# Inspect missings

### **Labeled ClinPGx dataset**

In [ ]:
## Delete rows that only contain NaN values

cols_to_check = [c for c in labeled_clinpgx_dataset.columns if c != 'User_input']

labeled_clinpgx_dataset_rmna = labeled_clinpgx_dataset[labeled_clinpgx_dataset[cols_to_check].notna().any(axis=1)]

In [ ]:
def missing_per_col(df):
  print(
    (df.isna().mean() * 100)
    .sort_values(ascending=False)
    .to_string()
    )

In [ ]:
missing_per_col(labeled_clinpgx_dataset_rmna)

MutPred_rankscore                       84.516880
EVE_score(class)                        83.818393
M-CAP_rankscore                         80.675204
regsnp_fpr                              80.442375
MVP_rankscore                           74.854482
InterVar_automated                      66.472643
MutationAssessor_rankscore              54.598370
DEOGEN2_rankscore                       51.105937
foldxDdg                                47.962747
foldxplddt                              47.962747
popEVE_score                            46.216531
Polyphen2_HVAR_pred                     45.983702
Polyphen2_HDIV_pred                     45.867288
MPC_rankscore                           44.237485
LRT_converted_rankscore                 44.004657
PrimateAI_rankscore                     43.538999
ESM1b_score                             42.840512
LIST-S2_rankscore                       42.724098
fathmm-XF_coding_rankscore              42.491269
Eigen-raw_coding_rankscore              41.909197


In [ ]:
## Delete columns with high percentage of missing values
## I chose 50% of missing values as the threshold after inspection of the % of missings per column

def del_high_missing_cols(df, threshold):
  cols_to_keep = (
    df.isna().mean().mul(100)
    .loc[lambda x: x < threshold]
    .index
    .tolist()
  )
  df_cleaned = df[cols_to_keep]
  return df_cleaned


In [ ]:
labeled_clinpgx_dataset_rmna = del_high_missing_cols(labeled_clinpgx_dataset_rmna, 50)

In [ ]:
def missing_per_row(df):
  missing_pct_per_row = df.isnull().mean(axis=1) * 100
  thresholds = [10, 20, 30, 50, 75, 90]
  for t in thresholds:
      bad_rows_count = (missing_pct_per_row > t).sum()

      pct_of_dataset = (bad_rows_count / len(df)) * 100

      print(f"{t}% missing: {bad_rows_count} rows ({pct_of_dataset:.2f}% of dataset)")



In [ ]:
missing_per_row(labeled_clinpgx_dataset_rmna)

10% missing: 381 rows (44.35% of dataset)
20% missing: 369 rows (42.96% of dataset)
30% missing: 368 rows (42.84% of dataset)
50% missing: 336 rows (39.12% of dataset)
75% missing: 178 rows (20.72% of dataset)
90% missing: 178 rows (20.72% of dataset)


In [ ]:
def rem_high_miss_pct_rows(df, max_missing_allowed):
  missing_pct_per_row = df.isnull().mean(axis=1) * 100
  df_cleaned = df[missing_pct_per_row <= max_missing_allowed]

  print(f"Original shape: {df.shape}")
  print(f"Cleaned shape: {df_cleaned.shape}")

  return df_cleaned


In [ ]:
labeled_clinpgx_dataset_rmna = rem_high_miss_pct_rows(labeled_clinpgx_dataset_rmna, 75)

Original shape: (859, 49)
Cleaned shape: (681, 49)


### **Unlabeled Populational dataset**

In [ ]:
## Delete rows that only contain NaN values

cols_to_check = [c for c in unlabeled_populational_dataset.columns if c != 'User_input']

unlabeled_populational_dataset_rmna = unlabeled_populational_dataset[unlabeled_populational_dataset[cols_to_check].notna().any(axis=1)]

In [ ]:
missing_per_col(unlabeled_populational_dataset_rmna)

InterVar_automated                      99.980450
regsnp_fpr                              99.957383
EVE_score(class)                        86.545080
MutPred_rankscore                       62.839369
popEVE_score                            47.001290
foldxplddt                              35.696192
foldxDdg                                35.696192
LRT_converted_rankscore                 21.151663
MPC_rankscore                           19.106343
MutationAssessor_rankscore              19.060632
M-CAP_rankscore                         18.323631
AlphaMissense_pathogenicity(class)      12.830297
FATHMM_converted_rankscore              12.586833
ESM1b_score                             12.543654
Conservation_score                      12.086966
pp_known_num_interactions               10.783711
pp_known_has_interactions               10.783711
num_pockets                             10.783711
CADD_phred_like_score                   10.783711
top_pocket_score                        10.783711


In [ ]:
unlabeled_populational_dataset_rmna = del_high_missing_cols(unlabeled_populational_dataset_rmna, 50)

In [ ]:
missing_per_row(unlabeled_populational_dataset_rmna)

10% missing: 155102 rows (21.81% of dataset)
20% missing: 115678 rows (16.27% of dataset)
30% missing: 106933 rows (15.04% of dataset)
50% missing: 39819 rows (5.60% of dataset)
75% missing: 6463 rows (0.91% of dataset)
90% missing: 2264 rows (0.32% of dataset)


In [ ]:
unlabeled_populational_dataset_rmna = rem_high_miss_pct_rows(unlabeled_populational_dataset_rmna, 75)

Original shape: (710989, 53)
Cleaned shape: (704526, 53)


# Final dataset

In [ ]:
labeled_clinpgx_dataset_rmna['label'] = 1
unlabeled_populational_dataset_rmna['label'] = -1

In [ ]:
common_cols = labeled_clinpgx_dataset_rmna.columns.intersection(
    unlabeled_populational_dataset_rmna.columns
    ).to_list()

In [ ]:
def subset_for_common(df):
  df = df[common_cols]
  return df


labeled_clinpgx_dataset_rmna = subset_for_common(labeled_clinpgx_dataset_rmna)
unlabeled_populational_dataset_rmna = subset_for_common(unlabeled_populational_dataset_rmna)

In [ ]:
full_dataset = pd.concat([labeled_clinpgx_dataset_rmna, unlabeled_populational_dataset_rmna], axis=0)

In [ ]:
full_dataset.to_csv('/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/ml_dataset.csv', index=False)

*   **v1: KNN Imputed**. KNN fills gaps based on the 5 most similar variants
*   **v2: MICE**: A highly robust machine learning approach that models each missing feature as a function of all the other features
*   **v3: Median Imputation**



####